In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train_final.csv')
test  = pd.read_csv('../data/test_final.csv')

top_features = ['TransactionAmt', 'card1', 'card6', 'C1', 'C2', 
                'C4', 'D10', 'card1_freq', 'card1_avg_amt', 'amt_diff_from_avg']

train_sample = train[top_features].sample(n=1000, random_state=42)
test_sample  = test[top_features].sample(n=500, random_state=42)

print(f"Reference (train): {train_sample.shape}")
print(f"Current (test):    {test_sample.shape}")

Reference (train): (1000, 10)
Current (test):    (500, 10)


In [3]:
# Dùng KS test để detect drift từng feature
drift_results = []

for col in top_features:
    stat, pvalue = stats.ks_2samp(
        train_sample[col].dropna(),
        test_sample[col].dropna()
    )
    drifted = pvalue < 0.05
    drift_results.append({
        'feature': col,
        'KS_stat': round(stat, 4),
        'p_value': round(pvalue, 4),
        'drifted': '⚠️ YES' if drifted else '✅ NO'
    })

df_drift = pd.DataFrame(drift_results)
print("📊 KẾT QUẢ DATA DRIFT (KS Test):")
print("="*55)
print(df_drift.to_string(index=False))

n_drifted = sum(1 for r in drift_results if '⚠️' in r['drifted'])
print(f"\nTổng drift: {n_drifted}/{len(top_features)} features ({n_drifted/len(top_features)*100:.0f}%)")

if n_drifted/len(top_features) > 0.3:
    print("⚠️  CẢNH BÁO: Cần retrain model!")
else:
    print("✅ Drift ở mức chấp nhận được")

📊 KẾT QUẢ DATA DRIFT (KS Test):
          feature  KS_stat  p_value drifted
   TransactionAmt    0.160   0.0000  ⚠️ YES
            card1    0.091   0.0078  ⚠️ YES
            card6    0.171   0.0000  ⚠️ YES
               C1    0.220   0.0000  ⚠️ YES
               C2    0.233   0.0000  ⚠️ YES
               C4    0.249   0.0000  ⚠️ YES
              D10    0.086   0.0141  ⚠️ YES
       card1_freq    0.104   0.0014  ⚠️ YES
    card1_avg_amt    0.036   0.7763    ✅ NO
amt_diff_from_avg    0.160   0.0000  ⚠️ YES

Tổng drift: 9/10 features (90%)
⚠️  CẢNH BÁO: Cần retrain model!


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Giả lập: fraud pattern thay đổi theo tháng
# Tháng 1: fraud bình thường
# Tháng 2: hacker tăng giao dịch lớn
# Tháng 3: hacker chuyển sang giao dịch nhỏ

np.random.seed(42)
months = ['Tháng 1', 'Tháng 2', 'Tháng 3', 'Tháng 4', 'Tháng 5', 'Tháng 6']

# Giả lập F1 score theo thời gian khi không retrain
f1_no_retrain   = [0.50, 0.47, 0.43, 0.38, 0.34, 0.29]
f1_with_retrain = [0.50, 0.47, 0.51, 0.49, 0.52, 0.50]

plt.figure(figsize=(10, 5))
plt.plot(months, f1_no_retrain,   'r-o', linewidth=2, label='Không retrain')
plt.plot(months, f1_with_retrain, 'g-o', linewidth=2, label='Có retrain khi drift')
plt.axhline(y=0.4, color='orange', linestyle='--', label='Ngưỡng cảnh báo (F1=0.4)')
plt.fill_between(range(len(months)), f1_no_retrain, f1_with_retrain, alpha=0.1, color='red')
plt.xticks(range(len(months)), months)
plt.ylabel('F1 Score')
plt.title('Model Performance Degradation Over Time\n(Concept Drift Simulation)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/drift_simulation.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu drift_simulation.png!")

✅ Đã lưu drift_simulation.png!


In [6]:
print("="*50)
print("📋 TÓM TẮT DRIFT ANALYSIS")
print("="*50)
print("""
1. DATA DRIFT: Phân phối features thay đổi giữa
   train và test → cần monitor liên tục

2. CONCEPT DRIFT: Fraud pattern thay đổi theo thời
   gian → model F1 giảm dần nếu không retrain

3. GIẢI PHÁP:
   - Monitor drift hàng tuần với Evidently
   - Retrain khi F1 < 0.4 hoặc drift > 30%
   - Lưu toàn bộ version với MLflow

4. Ý NGHĨA THỰC TẾ:
   - Đây là lý do hệ thống fraud detection
     THỰC TẾ cần continuous monitoring
   - Phân biệt đồ án này với các đồ án
     chỉ train 1 lần rồi thôi!
""")

📋 TÓM TẮT DRIFT ANALYSIS

1. DATA DRIFT: Phân phối features thay đổi giữa
   train và test → cần monitor liên tục

2. CONCEPT DRIFT: Fraud pattern thay đổi theo thời
   gian → model F1 giảm dần nếu không retrain

3. GIẢI PHÁP:
   - Monitor drift hàng tuần với Evidently
   - Retrain khi F1 < 0.4 hoặc drift > 30%
   - Lưu toàn bộ version với MLflow

4. Ý NGHĨA THỰC TẾ:
   - Đây là lý do hệ thống fraud detection
     THỰC TẾ cần continuous monitoring
   - Phân biệt đồ án này với các đồ án
     chỉ train 1 lần rồi thôi!

